# CS 577 Homework 3 Attention and Transformer Forward

**Student starter notebook**

Due November 8, 2026, 11:59 p.m. Central Time. Individual work; 8% of the
course grade. Read the assignment PDF before starting. Complete the eight
TODO functions in `hw3_attention.py` and the notebook TODOs below.
Before completion, NotImplementedError failures are expected. Before
submission, restart the kernel and run ALL cells with visible outputs on CPU.
Keep written calculations and interpretations in your LaTeX/PDF report.


In [1]:
import copy
import platform
import subprocess
import sys
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import hw3_attention as h

np.set_printoptions(precision=6, suppress=True)
torch.manual_seed(577)
torch.set_num_threads(1)
print({"Python": platform.python_version(), "NumPy": np.__version__,
       "PyTorch": torch.__version__, "Matplotlib": matplotlib.__version__,
       "device": "cpu", "seed": 577})

def max_error(a, b):
    return float(np.max(np.abs(np.asarray(a) - np.asarray(b))))


{'Python': '3.11.18', 'NumPy': '2.4.6', 'PyTorch': '2.15.0', 'Matplotlib': '3.11.2', 'device': 'cpu', 'seed': 577}


## 0 Setup and axis conventions

Put your axis meanings and the `(B,1,1,Tk)` mask explanation in the report.
This cell calls the public tests; run it after implementing the module.


In [2]:
result = subprocess.run([sys.executable, "-m", "unittest", "-v", "test_hw3_public.py"],
                        capture_output=True, text=True)
print(result.stdout)
print(result.stderr)
print("public-test return code:", result.returncode)
assert result.returncode == 0, "Complete/correct your functions before the final run-all."



test_01_stable_softmax_and_mask (test_hw3_public.AttentionTests.test_01_stable_softmax_and_mask) ... ok
test_02_softmax_last_axis_and_key_mask_broadcast (test_hw3_public.AttentionTests.test_02_softmax_last_axis_and_key_mask_broadcast) ... ok
test_03_causal_mask_includes_self (test_hw3_public.AttentionTests.test_03_causal_mask_includes_self) ... ok
test_04_attention_uniform_reads_and_rectangular_shapes (test_hw3_public.AttentionTests.test_04_attention_uniform_reads_and_rectangular_shapes) ... ok
test_05_hand_case_causal_visibility (test_hw3_public.AttentionTests.test_05_hand_case_causal_visibility) ... ok
test_06_head_axes_and_exact_roundtrip (test_hw3_public.HeadAndBlockTests.test_06_head_axes_and_exact_roundtrip) ... ok
test_07_mha_cache_and_identity_uniform_case (test_hw3_public.HeadAndBlockTests.test_07_mha_cache_and_identity_uniform_case) ... ok
test_08_mha_cross_attention_and_mask (test_hw3_public.HeadAndBlockTests.test_08_mha_cross_attention_and_mask) ... ok
test_09_position_vec

## 1 and 2 One attention calculation

Rows are The, cat, sleeps. First complete your hand calculation in the
written report, then use this cell to check full-precision values. The
returned score cache is scaled BEFORE masking; weights reflect the mask.


In [3]:
Q, K, V = h.hand_case()
Y_full, c_full = h.scaled_dot_product_attention(Q, K, V)
Y_causal, c_causal = h.scaled_dot_product_attention(Q, K, V, h.causal_mask(3))
print("QK transpose:\n", (Q @ K.swapaxes(-1, -2))[0, 0])
print("scaled scores:\n", c_full["scores"][0, 0])
print("full weights:\n", c_full["weights"][0, 0])
print("full outputs:\n", Y_full[0, 0])
print("causal allowed:\n", h.causal_mask(3))
print("causal weights:\n", c_causal["weights"][0, 0])
print("causal outputs:\n", Y_causal[0, 0])


QK transpose:
 [[1. 0. 1.]
 [0. 1. 1.]
 [1. 1. 2.]]
scaled scores:
 [[0.707107 0.       0.707107]
 [0.       0.707107 0.707107]
 [0.707107 0.707107 1.414214]]
full weights:
 [[0.401112 0.197776 0.401112]
 [0.197776 0.401112 0.401112]
 [0.248255 0.248255 0.50349 ]]
full outputs:
 [[1.604448 0.796664]
 [1.401112 1.203336]
 [1.758725 1.      ]]
causal allowed:
 [[ True False False]
 [ True  True False]
 [ True  True  True]]
causal weights:
 [[1.       0.       0.      ]
 [0.330238 0.669762 0.      ]
 [0.248255 0.248255 0.50349 ]]
causal outputs:
 [[1.       0.      ]
 [0.330238 1.339523]
 [1.758725 1.      ]]


## 2.1 Stability and a fully blocked row

This contract defines a zero-row guard when no key is allowed. It is not
ordinary softmax on all negative infinity. Explain your implementation
and its numerical behavior in the report.


In [4]:
scores_extreme = np.array([[1000., 999., -1000.], [1., 2., 3.]])
allowed_extreme = np.array([[True, True, False], [False, False, False]])
with np.errstate(over="raise", invalid="raise", divide="raise"):
    probabilities = h.masked_softmax(scores_extreme, allowed_extreme)
print("guarded weights:\n", probabilities)
print("row sums:", probabilities.sum(-1), "finite:", np.isfinite(probabilities).all())


guarded weights:
 [[0.731059 0.268941 0.      ]
 [0.       0.       0.      ]]
row sums: [1. 0.] finite: True


## 2.3 Padding and causal visibility

Two examples share storage length four. Actual lengths are two and four.
Implement the mask construction below. Padded query outputs are not used;
this check compares only valid queries. Print both combined masks and the
largest forbidden probability. The returned mask must be Boolean.


In [5]:
rng = np.random.default_rng(577)
Q_pad = rng.normal(size=(2, 2, 4, 2))
K_pad = rng.normal(size=(2, 2, 4, 2))
V_pad = rng.normal(size=(2, 2, 4, 3))
lengths = np.array([2, 4])


In [ ]:
# TODO: build key_valid:(B,T), key_allowed:(B,1,1,T),
# and combined_allowed:(B,1,T,T) with causal visibility.

raise NotImplementedError("Complete padding and causal mask construction")


NotImplementedError: Complete padding and causal mask construction

In [ ]:
padded_output, padded_cache = h.scaled_dot_product_attention(
    Q_pad, K_pad, V_pad, combined_allowed)
altered_values = V_pad.copy()
altered_values[0, :, 2:, :] *= 1e6
changed_output, _ = h.scaled_dot_product_attention(
    Q_pad, K_pad, altered_values, combined_allowed)
visible = np.broadcast_to(combined_allowed, padded_cache["weights"].shape)
forbidden_max = float(np.max(padded_cache["weights"][~visible]))
errors = [max_error(padded_output[b, :, :n], changed_output[b, :, :n])
          for b, n in enumerate(lengths)]
print("combined masks:\n", combined_allowed[:, 0].astype(int))
print("largest forbidden weight:", forbidden_max)
print("valid-query changes after padding perturbation:", errors)
assert forbidden_max == 0 and max(errors) < 1e-9


## 3.1 Multiple heads on the same token embeddings

The parameter helper supplies four dense, full-width projections. They
are sample random parameters, not trained language representations.
Report the shapes, weights and outputs, then interpret the maps.


In [ ]:
X = h.token_case()
mha_params = h.initialize_mha(4, seed=577)
Y_two, c_two = h.multi_head_attention(X, None, mha_params, 2)
print("input:", X.shape)
print("projected full width:", (X @ mha_params["Wq"]).shape)
print("cache shapes:", {key: value.shape for key, value in c_two.items()})
print("head 0 weights:\n", c_two["weights"][0, 0])
print("head 1 weights:\n", c_two["weights"][0, 1])
print("final MHA output:\n", Y_two[0])

fig, axes = plt.subplots(1, 2, figsize=(8.6, 3.8), constrained_layout=True)
words = ["The", "cat", "sleeps"]
for head, ax in enumerate(axes):
    im = ax.imshow(c_two["weights"][0, head], vmin=0, vmax=1, cmap="Blues")
    ax.set_xticks(range(3), words)
    ax.set_yticks(range(3), words)
    ax.set_xlabel("Key token"); ax.set_ylabel("Query token")
    ax.set_title(f"Head {head}")
fig.colorbar(im, ax=axes, label="Attention weight")
fig.savefig("hw3_attention_maps.png", dpi=180)
plt.show()


## 3.2 Independent PyTorch reference

Complete the function below using nn.MultiheadAttention. Map parameters
exactly as stated in the PDF: transpose each NumPy matrix, concatenate
Q/K/V input weights vertically, and transpose the output projection.
Use CPU float64, no biases/dropout, batch_first=True, and request per-head
weights. For this function allowed is None or a 2D (Tq,Tk) mask.
nn.MultiheadAttention uses True=blocked, so invert an allowed mask.


In [ ]:
def torch_mha_reference(X_query, X_context, params, num_heads, allowed=None):
    # TODO: return (NumPy output, NumPy per-head weights) from a faithful
    # independent torch.nn.MultiheadAttention module.
    raise NotImplementedError("Build the independent PyTorch MHA reference")


In [ ]:
reference_checks = []
for name, query, context, allowed in [
    ("unmasked self", X, None, None),
    ("causal self", X, None, h.causal_mask(3)),
    ("rectangular cross", X[:, :2], X, None),
]:
    ours, cached = h.multi_head_attention(query, context, mha_params, 2, allowed)
    reference, reference_weights = torch_mha_reference(
        query, context, mha_params, 2, allowed)
    errors = (max_error(ours, reference), max_error(cached["weights"], reference_weights))
    reference_checks.append((name, *errors))
print("case | maximum output error | maximum per-head weight error")
for row in reference_checks:
    print(row)
assert max(max(row[1:]) for row in reference_checks) < 1e-9


## 3.3 Head count and removing one head output

Compare fixed total width and fixed projections at H=1 and H=2. Then
independently remove each head output BEFORE the output projection.
This is an ablation of output features, not a modification of token masks.
Interpret the actual changes; do not claim a semantic function from
untrained maps or improved accuracy from this forward-only experiment.


In [ ]:
Y_one, c_one = h.multi_head_attention(X, None, mha_params, 1)
count = sum(p.size for p in mha_params.values())
print("H=1: features/head", c_one["Q"].shape[-1], "scores", c_one["scores"].shape,
      "parameters", count)
print("H=2: features/head", c_two["Q"].shape[-1], "scores", c_two["scores"].shape,
      "parameters", count)
print("maximum H=1 vs H=2 output difference:", max_error(Y_one, Y_two))


In [ ]:
# TODO: remove each head output separately and report the two maximum
# output changes after merging and applying Wo. Work on copies.
raise NotImplementedError("Complete the head-output ablation")


## 4.1 One pre-norm Transformer block

Use the specified pre-norm/ReLU order with provided LayerNorm.
Independently reproduce the same block in PyTorch below; keep normalization
axis, epsilon, parameters, residual paths and mask exactly matched.


In [ ]:
block_params = h.initialize_block(4, 8, seed=577)
Z, block_cache = h.transformer_block(X, block_params, 2)
print("block cache shapes:", {key: value.shape for key, value in block_cache.items()})
print("block output:\n", Z[0])


In [ ]:
def torch_block_reference(X, params, num_heads, allowed=None):
    # TODO: recreate the SAME pre-norm/ReLU block with PyTorch operations
    # and your independent torch_mha_reference. Return a NumPy array.
    raise NotImplementedError("Build the independent PyTorch block reference")


In [ ]:
Z_reference = torch_block_reference(X, block_params, 2)
print("NumPy vs PyTorch block max error:", max_error(Z, Z_reference))
assert max_error(Z, Z_reference) < 1e-9
zero_updates = copy.deepcopy(block_params)
zero_updates["mha"]["Wo"][:] = 0.
zero_updates["W2"][:] = 0.
zero_updates["b2"][:] = 0.
Z_zero, _ = h.transformer_block(X, zero_updates, 2)
print("zero-update residual max error:", max_error(Z_zero, X))
assert max_error(Z_zero, X) < 1e-9


## 4.2 Position vectors and reordered tokens

PE is added feature by feature, retaining shape (B,T,D). In the second
comparison, token embeddings move but position vectors stay in their slots.
Use no causal mask in this experiment. Explain the TWO reported errors.


In [ ]:
PE = h.sinusoidal_positions(3, 4)
order = np.array([2, 0, 1])
X_position = X + PE[None]
Z_reordered, _ = h.transformer_block(X[:, order], block_params, 2)
no_position_error = max_error(Z_reordered, Z[:, order])
Z_position, _ = h.transformer_block(X_position, block_params, 2)
Z_reordered_position, _ = h.transformer_block(X[:, order] + PE[None], block_params, 2)
position_error = max_error(Z_reordered_position, Z_position[:, order])
print("position vectors:\n", PE)
print("input plus positions:\n", X_position[0])
print("reordering error without positions:", no_position_error)
print("reordering error with positions attached to slots:", position_error)
assert no_position_error < 1e-9


## 4.3 Change only the last input

Both runs use the same fixed position vectors. Compare the first TWO
output positions, first with causal attention and then with all keys visible.
Explain how the allowed attention paths account for the measured change.


In [ ]:
X_changed = X_position.copy()
X_changed[:, 2] += np.array([5., -3., 2., -4.])
prefix_errors = {}
for name, allowed in [("causal", h.causal_mask(3)), ("unmasked", None)]:
    original, _ = h.transformer_block(X_position, block_params, 2, allowed)
    altered, _ = h.transformer_block(X_changed, block_params, 2, allowed)
    prefix_errors[name] = max_error(original[:, :2], altered[:, :2])
print("first-two-position maximum errors:", prefix_errors)
assert prefix_errors["causal"] < 1e-9


## 5 Written architecture comparison and disclosure

In the report, trace BOS -> Le and Le -> chat in encoder-decoder generation.
Define the vectors used for queries and source keys/values. Then trace the
final prompt position in decoder-only generation and explain per-query
causal visibility during prefill. BOS is supplied in the stated example.

Include your hand calculation, measured-error tables, labeled attention
maps, head comparisons, position/causal interpretations, and assistance/AI
disclosure. The report should be understandable without guessing new symbols.
